# PERSUADE: Within-Essay Stability Analysis

## Question
Are shape metrics (especially `log_slope_local`) stable **within** individual essays across different segments?

If shape metrics are consistent across an essay's 20 windows, the essay has a distinctive "profile" - potentially a signature of the writer's memory-use pattern.

## Approach
1. **Compute per-window shape metrics** (not just gains, but the derived shape metrics)
2. **Within-essay ICC**: Var(between-essay) / Var(total) - how much is essay-level vs window noise?
3. **Spatial stability**: First-half windows vs second-half windows correlation
4. **Profile consistency**: Can we identify essays by their shape profile?

## Interpretation
- High ICC = essays have distinctive, stable profiles
- Low ICC = shape metrics are mostly noise, no stable "signature"

## Data
Uses window-level results from `local_lag_curve_sliding_v1/`

In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scipy statsmodels

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

print("Libraries loaded")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
RESULTS_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade/local_lag_curve_sliding_v1"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"

EXPERIMENT = 'within_essay_stability_v1'

# K values from original analysis
K_GRID = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]

# Reproducibility
RANDOM_SEED = 42
N_BOOTSTRAP = 100

# Create output directory
output_dir = Path(f"{OUTPUT_BASE}/{EXPERIMENT}")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT}")
print(f"Loading from: {RESULTS_BASE}")
print(f"Output to: {output_dir}")

## 1. Load Data

In [ ]:
# Load window-level results
df_windows = pd.read_csv(f"{RESULTS_BASE}/window_level_results_local_lag.csv")
print(f"Window-level data: {df_windows.shape}")
print(f"Columns: {list(df_windows.columns)}")

# Load essay-level results for metadata
df_essays = pd.read_csv(f"{RESULTS_BASE}/essay_level_results_local_lag.csv")
print(f"\nEssay-level data: {df_essays.shape}")

# Load cohort to get prompt_id
COHORT_PATH = "/content/drive/MyDrive/LRTIA/Data/persuade_clean/cohorts/persuade_score_long_cohort.jsonl"
cohort_data = []
with open(COHORT_PATH, 'r') as f:
    for line in f:
        if line.strip():
            cohort_data.append(json.loads(line))
df_cohort = pd.DataFrame(cohort_data)[['essay_id', 'prompt_id']]
print(f"\nCohort data (for prompt_id): {df_cohort.shape}")
print(f"Unique prompts: {df_cohort['prompt_id'].nunique()}")

In [ ]:
# Check structure
print("Window-level sample:")
print(df_windows.head(15))

n_essays = df_windows['essay_id'].nunique()
n_windows_per_essay = df_windows.groupby('essay_id')['window_idx'].nunique().median()
n_k_per_window = df_windows.groupby(['essay_id', 'window_idx'])['k'].nunique().median()

print(f"\nUnique essays: {n_essays}")
print(f"Windows per essay (median): {n_windows_per_essay}")
print(f"K values per window: {n_k_per_window}")

## 2. Compute Per-Window Shape Metrics

The window-level data has `gain_k` for each k. We need to compute shape metrics for each window.

In [ ]:
def compute_window_shape_metrics(window_df):
    """
    Compute shape metrics for a single window from its gain values.
    Input: DataFrame with columns ['k', 'gain_k'] for one window
    """
    # Convert to dict for easier access
    gains = dict(zip(window_df['k'], window_df['gain_k']))
    
    results = {}
    
    # early_ratio = gain_16 / gain_128
    if gains.get(128, 0) > 0:
        results['early_ratio'] = gains.get(16, 0) / gains[128]
    else:
        results['early_ratio'] = np.nan
    
    # log_slope_local: slope of gain vs log(k) for k >= 2
    ks_for_fit = [k for k in K_GRID if k >= 2 and k in gains]
    if len(ks_for_fit) >= 2:
        log_ks = [np.log(k + 1) for k in ks_for_fit]
        gains_for_fit = [gains[k] for k in ks_for_fit]
        slope, intercept, r_value, p_value, std_err = stats.linregress(log_ks, gains_for_fit)
        results['log_slope_local'] = slope
        results['log_slope_r2'] = r_value ** 2
    else:
        results['log_slope_local'] = np.nan
        results['log_slope_r2'] = np.nan
    
    # AUC on log scale
    auc_log = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        if k1 in gains and k2 in gains:
            g1, g2 = gains[k1], gains[k2]
            width = np.log(k2 + 1) - np.log(k1 + 1)
            auc_log += 0.5 * (g1 + g2) * width
    results['auc_log_k'] = auc_log
    
    # AUC on linear scale
    auc_linear = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        if k1 in gains and k2 in gains:
            g1, g2 = gains[k1], gains[k2]
            width = k2 - k1
            auc_linear += 0.5 * (g1 + g2) * width
    results['auc_linear_k'] = auc_linear
    
    # Total gain (fluency proxy)
    results['gain_128'] = gains.get(128, np.nan)
    
    return results


print("Computing per-window shape metrics...")

# Compute for each window
window_metrics = []

for (essay_id, window_idx), group in df_windows.groupby(['essay_id', 'window_idx']):
    metrics = compute_window_shape_metrics(group)
    metrics['essay_id'] = essay_id
    metrics['window_idx'] = window_idx
    metrics['target_start'] = group['target_start'].iloc[0]
    window_metrics.append(metrics)

df_window_metrics = pd.DataFrame(window_metrics)
print(f"\nPer-window metrics: {df_window_metrics.shape}")
print(df_window_metrics.head())

In [ ]:
# Add essay-level metadata (including prompt_id)
essay_meta = df_essays[['essay_id', 'score_bin', 'grade', 'token_count', 'baseline_nll']].copy()
essay_meta = essay_meta.merge(df_cohort, on='essay_id', how='left')

df_window_metrics = df_window_metrics.merge(essay_meta, on='essay_id', how='left')

print(f"Window metrics with metadata: {df_window_metrics.shape}")
print(f"Prompts in data: {df_window_metrics['prompt_id'].nunique()}")
print(f"\nMetric distributions:")
for col in ['log_slope_local', 'auc_linear_k', 'early_ratio', 'gain_128']:
    data = df_window_metrics[col].dropna()
    print(f"  {col}: mean={data.mean():.4f}, std={data.std():.4f}, n={len(data)}")

## 3. Within-Essay ICC (Intraclass Correlation)

ICC = Var(between-essay) / Var(total)

High ICC means essays have distinctive, stable profiles across their windows.

In [ ]:
def compute_icc_oneway(df, metric_col, group_col):
    """
    Compute ICC(1,1) using one-way random effects ANOVA.
    ICC = (MS_between - MS_within) / (MS_between + (k-1)*MS_within)
    where k is the average group size.
    """
    data = df[[group_col, metric_col]].dropna()
    
    groups = data.groupby(group_col)[metric_col]
    n_groups = groups.ngroups
    group_means = groups.mean()
    group_sizes = groups.size()
    grand_mean = data[metric_col].mean()
    
    # Between-group sum of squares
    ss_between = sum(n * (m - grand_mean)**2 for n, m in zip(group_sizes, group_means))
    df_between = n_groups - 1
    
    # Within-group sum of squares
    ss_within = sum(((g - g.mean())**2).sum() for _, g in groups)
    df_within = len(data) - n_groups
    
    # Mean squares
    ms_between = ss_between / df_between if df_between > 0 else 0
    ms_within = ss_within / df_within if df_within > 0 else 0
    
    # Average group size
    k = len(data) / n_groups
    
    # ICC(1,1)
    if ms_between + (k - 1) * ms_within > 0:
        icc = (ms_between - ms_within) / (ms_between + (k - 1) * ms_within)
    else:
        icc = 0
    
    # Variance components
    var_between = max(0, (ms_between - ms_within) / k)
    var_within = ms_within
    var_total = var_between + var_within
    
    return {
        'icc': icc,
        'var_between': var_between,
        'var_within': var_within,
        'var_total': var_total,
        'pct_between': 100 * var_between / var_total if var_total > 0 else 0,
        'n_groups': n_groups,
        'n_obs': len(data),
        'k_avg': k,
        'ms_between': ms_between,
        'ms_within': ms_within,
    }


print("="*80)
print("WITHIN-ESSAY ICC: How stable are shape metrics across windows?")
print("="*80)
print("\nICC = fraction of variance that is between-essay (vs within-essay noise)")
print("High ICC = essays have distinctive, stable profiles")
print("\nRule of thumb: <0.10 weak, 0.10-0.30 moderate, >0.30 strong\n")

icc_results = {}

# Primary shape metrics (focus on reliable ones)
metrics_to_analyze = [
    ('log_slope_local', 'PRIMARY - best reliability'),
    ('auc_linear_k', 'PRIMARY - secondary shape'),
    ('gain_128', 'SECONDARY - fluency/overall benefit'),
    ('early_ratio', 'EXPLORATORY - lower reliability'),
    ('auc_log_k', 'EXPLORATORY - lower reliability'),
]

print(f"{'Metric':<20} {'ICC':>8} {'%Between':>10} {'%Within':>10} {'Interpret':>12}")
print("-"*65)

for metric, label in metrics_to_analyze:
    result = compute_icc_oneway(df_window_metrics, metric, 'essay_id')
    icc_results[metric] = result
    
    icc = result['icc']
    if icc < 0.10:
        interp = "weak"
    elif icc < 0.30:
        interp = "moderate"
    else:
        interp = "STRONG"
    
    print(f"{metric:<20} {icc:>8.3f} {result['pct_between']:>9.1f}% {100-result['pct_between']:>9.1f}% {interp:>12}")

print(f"\nN essays: {icc_results['log_slope_local']['n_groups']}")
print(f"Avg windows per essay: {icc_results['log_slope_local']['k_avg']:.1f}")

In [ ]:
# Visualize variance components
fig, ax = plt.subplots(figsize=(10, 6))

metrics = [m for m, _ in metrics_to_analyze]
pct_between = [icc_results[m]['pct_between'] for m in metrics]
pct_within = [100 - icc_results[m]['pct_between'] for m in metrics]

x = np.arange(len(metrics))
width = 0.6

bars1 = ax.bar(x, pct_between, width, label='Between-Essay (Signal)', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x, pct_within, width, bottom=pct_between, label='Within-Essay (Noise)', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Variance (%)', fontsize=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_title('Variance Decomposition: Between-Essay vs Within-Essay\n(Higher green = more stable "signature")', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.set_ylim(0, 100)

# Add ICC values as text
for i, m in enumerate(metrics):
    icc = icc_results[m]['icc']
    ax.text(i, 5, f'ICC={icc:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig(output_dir / 'icc_variance_components.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Spatial Stability: First-Half vs Second-Half Windows

Do windows from the first half of the essay correlate with windows from the second half?

In [ ]:
print("="*80)
print("SPATIAL STABILITY: First-Half vs Second-Half Windows")
print("="*80)
print("\nFor each essay, split windows into first-half (by position) vs second-half.")
print("Compute mean metric for each half, then correlate across essays.\n")

def compute_spatial_split_half(df, metric):
    """Split windows by position (first half vs second half of essay)."""
    first_half_means = []
    second_half_means = []
    
    for essay_id, group in df.groupby('essay_id'):
        # Sort by position
        group = group.sort_values('target_start')
        n = len(group)
        if n < 4:  # Need at least 2 per half
            continue
        
        mid = n // 2
        first_half = group.iloc[:mid][metric].dropna()
        second_half = group.iloc[mid:][metric].dropna()
        
        if len(first_half) > 0 and len(second_half) > 0:
            first_half_means.append(first_half.mean())
            second_half_means.append(second_half.mean())
    
    if len(first_half_means) > 2:
        r_pearson, p_pearson = stats.pearsonr(first_half_means, second_half_means)
        r_spearman, p_spearman = stats.spearmanr(first_half_means, second_half_means)
        return {
            'r_pearson': r_pearson,
            'p_pearson': p_pearson,
            'r_spearman': r_spearman,
            'p_spearman': p_spearman,
            'n_essays': len(first_half_means),
            'first_half_means': first_half_means,
            'second_half_means': second_half_means,
        }
    return None


spatial_results = {}

print(f"{'Metric':<20} {'r (Pearson)':>12} {'r (Spearman)':>14} {'N essays':>10}")
print("-"*60)

for metric, label in metrics_to_analyze:
    result = compute_spatial_split_half(df_window_metrics, metric)
    if result:
        spatial_results[metric] = result
        print(f"{metric:<20} {result['r_pearson']:>12.3f} {result['r_spearman']:>14.3f} {result['n_essays']:>10}")
    else:
        print(f"{metric:<20} {'N/A':>12}")

In [ ]:
# Plot spatial stability for primary metric
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['log_slope_local', 'auc_linear_k']):
    if metric in spatial_results:
        result = spatial_results[metric]
        ax.scatter(result['first_half_means'], result['second_half_means'], alpha=0.5, s=20)
        
        # Add diagonal
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'r--', alpha=0.5, label='y=x')
        
        ax.set_xlabel(f'{metric} (First Half)', fontsize=11)
        ax.set_ylabel(f'{metric} (Second Half)', fontsize=11)
        ax.set_title(f'Spatial Stability: {metric}\nr={result["r_pearson"]:.3f}', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'spatial_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Within vs Between Essay Distances

Are within-essay differences smaller than between-essay differences?

In [ ]:
print("="*80)
print("WITHIN vs BETWEEN ESSAY DISTANCES")
print("="*80)

def compute_within_between_distances(df, metric, n_sample=1000):
    """
    Compute distributions of within-essay and between-essay absolute differences.
    """
    rng = np.random.default_rng(RANDOM_SEED)
    
    # Within-essay: sample pairs of windows from same essay
    within_diffs = []
    essays_with_multiple = df.groupby('essay_id').filter(lambda x: len(x) >= 2)['essay_id'].unique()
    
    for essay_id in rng.choice(essays_with_multiple, min(n_sample, len(essays_with_multiple)), replace=False):
        vals = df[df['essay_id'] == essay_id][metric].dropna().values
        if len(vals) >= 2:
            # Random pair
            idx = rng.choice(len(vals), 2, replace=False)
            within_diffs.append(abs(vals[idx[0]] - vals[idx[1]]))
    
    # Between-essay: sample pairs of windows from different essays
    between_diffs = []
    all_data = df[['essay_id', metric]].dropna()
    
    for _ in range(n_sample):
        # Sample two different essays
        essays = rng.choice(all_data['essay_id'].unique(), 2, replace=False)
        val1 = all_data[all_data['essay_id'] == essays[0]][metric].values
        val2 = all_data[all_data['essay_id'] == essays[1]][metric].values
        if len(val1) > 0 and len(val2) > 0:
            between_diffs.append(abs(rng.choice(val1) - rng.choice(val2)))
    
    return np.array(within_diffs), np.array(between_diffs)


distance_results = {}

print(f"\n{'Metric':<20} {'Within (mean)':>14} {'Between (mean)':>15} {'Ratio':>10} {'p-value':>12}")
print("-"*75)

for metric, label in metrics_to_analyze:
    within, between = compute_within_between_distances(df_window_metrics, metric)
    
    if len(within) > 10 and len(between) > 10:
        # Mann-Whitney U test
        stat, pval = stats.mannwhitneyu(within, between, alternative='less')
        ratio = np.mean(within) / np.mean(between) if np.mean(between) > 0 else np.nan
        
        distance_results[metric] = {
            'within': within,
            'between': between,
            'within_mean': np.mean(within),
            'between_mean': np.mean(between),
            'ratio': ratio,
            'pval': pval,
        }
        
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"{metric:<20} {np.mean(within):>14.4f} {np.mean(between):>15.4f} {ratio:>10.3f} {pval:>11.4f}{sig}")

print("\n* p<0.05, ** p<0.01, *** p<0.001 (one-tailed: within < between)")
print("Ratio < 1 means within-essay distances are smaller (good for signatures)")

In [ ]:
# Plot within vs between distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['log_slope_local', 'auc_linear_k']):
    if metric in distance_results:
        result = distance_results[metric]
        
        ax.hist(result['within'], bins=50, alpha=0.6, label=f'Within-essay (mean={result["within_mean"]:.3f})', 
                color='#2ecc71', density=True)
        ax.hist(result['between'], bins=50, alpha=0.6, label=f'Between-essay (mean={result["between_mean"]:.3f})', 
                color='#e74c3c', density=True)
        
        ax.axvline(result['within_mean'], color='#27ae60', linestyle='--', linewidth=2)
        ax.axvline(result['between_mean'], color='#c0392b', linestyle='--', linewidth=2)
        
        ax.set_xlabel(f'Absolute Difference in {metric}', fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.set_title(f'Within vs Between Essay Distances\n{metric} (ratio={result["ratio"]:.2f})', 
                     fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'within_vs_between_distances.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Bootstrap Rank Stability

If we split each essay's windows into two random halves, do the essay rankings correlate?

In [ ]:
print("="*80)
print(f"BOOTSTRAP RANK STABILITY ({N_BOOTSTRAP} iterations)")
print("="*80)
print("\nSplit each essay's windows into random halves, compute mean for each half.")
print("Correlate essay rankings from half A vs half B.\n")

def bootstrap_rank_stability(df, metric, n_iters=100):
    """Bootstrap split-half rank correlations."""
    rng = np.random.default_rng(RANDOM_SEED)
    
    # Filter to essays with enough windows
    essay_windows = df.groupby('essay_id').filter(lambda x: x[metric].dropna().shape[0] >= 4)
    essay_ids = essay_windows['essay_id'].unique()
    
    pearson_rs = []
    spearman_rs = []
    
    for _ in range(n_iters):
        half_a_means = []
        half_b_means = []
        
        for essay_id in essay_ids:
            vals = essay_windows[essay_windows['essay_id'] == essay_id][metric].dropna().values
            if len(vals) >= 4:
                idx = rng.permutation(len(vals))
                mid = len(idx) // 2
                half_a_means.append(np.mean(vals[idx[:mid]]))
                half_b_means.append(np.mean(vals[idx[mid:]]))
        
        if len(half_a_means) > 2:
            r_p, _ = stats.pearsonr(half_a_means, half_b_means)
            r_s, _ = stats.spearmanr(half_a_means, half_b_means)
            pearson_rs.append(r_p)
            spearman_rs.append(r_s)
    
    return {
        'pearson_rs': pearson_rs,
        'spearman_rs': spearman_rs,
        'pearson_mean': np.mean(pearson_rs),
        'pearson_ci': (np.percentile(pearson_rs, 2.5), np.percentile(pearson_rs, 97.5)),
        'spearman_mean': np.mean(spearman_rs),
        'spearman_ci': (np.percentile(spearman_rs, 2.5), np.percentile(spearman_rs, 97.5)),
        'n_essays': len(essay_ids),
    }


rank_results = {}

print(f"{'Metric':<20} {'r (Pearson)':>12} {'95% CI':>18} {'r (Spearman)':>14}")
print("-"*70)

for metric, label in metrics_to_analyze:
    result = bootstrap_rank_stability(df_window_metrics, metric, N_BOOTSTRAP)
    rank_results[metric] = result
    
    ci_str = f"[{result['pearson_ci'][0]:.3f}, {result['pearson_ci'][1]:.3f}]"
    print(f"{metric:<20} {result['pearson_mean']:>12.3f} {ci_str:>18} {result['spearman_mean']:>14.3f}")

print(f"\nN essays: {rank_results['log_slope_local']['n_essays']}")

In [ ]:
# Plot rank stability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['log_slope_local', 'auc_linear_k']):
    if metric in rank_results:
        result = rank_results[metric]
        
        ax.hist(result['pearson_rs'], bins=30, alpha=0.7, color='#3498db', edgecolor='black')
        ax.axvline(result['pearson_mean'], color='red', linestyle='--', linewidth=2, 
                   label=f'Mean r = {result["pearson_mean"]:.3f}')
        ax.axvline(0.7, color='green', linestyle=':', linewidth=2, label='Acceptable (0.7)')
        
        ax.set_xlabel('Split-Half Correlation (r)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'Rank Stability: {metric}\n({N_BOOTSTRAP} bootstrap iterations)', 
                     fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig(output_dir / 'rank_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Residualized Analysis (Control for Length/Fluency)

Is the signature real, or just that some essays are longer/easier-to-predict?

In [ ]:
print("="*80)
print("RESIDUALIZED ANALYSIS: Controlling for Length and Fluency")
print("="*80)
print("\nRegress shape metrics on token_count + baseline_nll, then test stability on residuals.")
print("This checks if the 'signature' is real vs just length/fluency effects.\n")

# Compute residuals for each metric
df_resid = df_window_metrics.copy()

for metric, _ in metrics_to_analyze:
    # Use essay-level covariates
    valid_mask = df_resid[metric].notna() & df_resid['token_count'].notna() & df_resid['baseline_nll'].notna()
    
    if valid_mask.sum() > 10:
        formula = f'{metric} ~ token_count + baseline_nll'
        model = smf.ols(formula, data=df_resid[valid_mask]).fit()
        
        # Store residuals
        resid_col = f'{metric}_resid'
        df_resid[resid_col] = np.nan
        df_resid.loc[valid_mask, resid_col] = model.resid
        
        print(f"{metric}: R² from covariates = {model.rsquared:.3f}")

print("\n" + "-"*60)
print("ICC on RESIDUALIZED metrics:")
print("-"*60)

icc_resid_results = {}

print(f"\n{'Metric':<25} {'ICC (raw)':>12} {'ICC (resid)':>12} {'Change':>10}")
print("-"*65)

for metric, label in metrics_to_analyze:
    resid_col = f'{metric}_resid'
    if resid_col in df_resid.columns:
        result_resid = compute_icc_oneway(df_resid, resid_col, 'essay_id')
        icc_resid_results[metric] = result_resid
        
        icc_raw = icc_results[metric]['icc']
        icc_resid = result_resid['icc']
        change = icc_resid - icc_raw
        
        print(f"{metric:<25} {icc_raw:>12.3f} {icc_resid:>12.3f} {change:>+10.3f}")

print("\nIf ICC(resid) ≈ ICC(raw), the signature is real (not just length/fluency).")
print("If ICC(resid) << ICC(raw), the 'signature' was mostly confounded.")

## 7b. Prompt ICC Analysis

Does prompt (topic) explain variance in shape metrics? If prompt ICC is ~0 (or vanishes after controls), we can drop "prompt" as an explanatory lever.

In [ ]:
print("="*80)
print("PROMPT ICC ANALYSIS: Does topic explain shape metric variance?")
print("="*80)

# First, aggregate to essay level for prompt analysis
df_essay_metrics = df_window_metrics.groupby('essay_id').agg({
    'log_slope_local': 'mean',
    'auc_linear_k': 'mean',
    'gain_128': 'mean',
    'early_ratio': 'mean',
    'auc_log_k': 'mean',
    'token_count': 'first',
    'baseline_nll': 'first',
    'prompt_id': 'first',
    'score_bin': 'first',
}).reset_index()

print(f"\nEssay-level data: {len(df_essay_metrics)} essays")
print(f"Prompts: {df_essay_metrics['prompt_id'].nunique()}")
print(f"\nEssays per prompt:")
print(df_essay_metrics['prompt_id'].value_counts())

# 1. UNCONDITIONAL PROMPT ICC
print("\n" + "="*60)
print("1. UNCONDITIONAL PROMPT ICC")
print("="*60)
print("What % of variance is attributable to prompt?\n")

prompt_icc_results = {}

print(f"{'Metric':<20} {'ICC_prompt':>12} {'%Prompt':>10} {'Interpret':>12}")
print("-"*60)

for metric, _ in metrics_to_analyze:
    result = compute_icc_oneway(df_essay_metrics, metric, 'prompt_id')
    prompt_icc_results[metric] = result
    
    icc = result['icc']
    if icc < 0.05:
        interp = "negligible"
    elif icc < 0.10:
        interp = "weak"
    elif icc < 0.20:
        interp = "moderate"
    else:
        interp = "substantial"
    
    print(f"{metric:<20} {icc:>12.3f} {result['pct_between']:>9.1f}% {interp:>12}")

In [ ]:
# 2. PROMPT ICC AFTER CONTROLLING FOR LENGTH + FLUENCY
print("\n" + "="*60)
print("2. PROMPT ICC AFTER CONTROLLING FOR LENGTH + FLUENCY")
print("="*60)
print("If prompt ICC collapses after fluency control, prompt is mostly topic/vocab, not memory-structure.\n")

# Compute residuals controlling for length and fluency
df_essay_resid = df_essay_metrics.copy()

for metric, _ in metrics_to_analyze:
    valid_mask = (df_essay_resid[metric].notna() & 
                  df_essay_resid['token_count'].notna() & 
                  df_essay_resid['baseline_nll'].notna())
    
    if valid_mask.sum() > 10:
        formula = f'{metric} ~ token_count + baseline_nll'
        model = smf.ols(formula, data=df_essay_resid[valid_mask]).fit()
        
        resid_col = f'{metric}_resid'
        df_essay_resid[resid_col] = np.nan
        df_essay_resid.loc[valid_mask, resid_col] = model.resid

# Compute prompt ICC on residuals
prompt_icc_resid_results = {}

print(f"{'Metric':<20} {'ICC (raw)':>12} {'ICC (resid)':>12} {'Change':>10} {'Interpretation':>20}")
print("-"*80)

for metric, _ in metrics_to_analyze:
    resid_col = f'{metric}_resid'
    if resid_col in df_essay_resid.columns:
        result_resid = compute_icc_oneway(df_essay_resid, resid_col, 'prompt_id')
        prompt_icc_resid_results[metric] = result_resid
        
        icc_raw = prompt_icc_results[metric]['icc']
        icc_resid = result_resid['icc']
        change = icc_resid - icc_raw
        
        if icc_resid < 0.02:
            interp = "VANISHED - drop prompt"
        elif abs(change) < 0.02:
            interp = "stable - real effect"
        elif change < -0.05:
            interp = "reduced - partial confound"
        else:
            interp = "unchanged"
        
        print(f"{metric:<20} {icc_raw:>12.3f} {icc_resid:>12.3f} {change:>+10.3f} {interp:>20}")

In [ ]:
# 3. PROMPT FIXED-EFFECT SPREAD (sanity check)
print("\n" + "="*60)
print("3. PROMPT FIXED-EFFECT SPREAD (min-max prompt means)")
print("="*60)
print("How much do prompt means actually vary?\n")

print(f"{'Metric':<20} {'Min prompt':>12} {'Max prompt':>12} {'Spread':>10} {'Grand mean':>12}")
print("-"*70)

prompt_spread_results = {}

for metric, _ in metrics_to_analyze:
    prompt_means = df_essay_metrics.groupby('prompt_id')[metric].mean()
    grand_mean = df_essay_metrics[metric].mean()
    
    min_val = prompt_means.min()
    max_val = prompt_means.max()
    spread = max_val - min_val
    
    prompt_spread_results[metric] = {
        'min': min_val,
        'max': max_val,
        'spread': spread,
        'grand_mean': grand_mean,
        'prompt_means': prompt_means,
    }
    
    print(f"{metric:<20} {min_val:>12.4f} {max_val:>12.4f} {spread:>10.4f} {grand_mean:>12.4f}")

# Show prompts with extreme values for primary metric
print("\n" + "-"*60)
print("Prompt means for log_slope_local (sorted):")
print("-"*60)
sorted_prompts = prompt_spread_results['log_slope_local']['prompt_means'].sort_values()
for prompt, val in sorted_prompts.items():
    n = len(df_essay_metrics[df_essay_metrics['prompt_id'] == prompt])
    print(f"  {val:.4f}  {prompt} (n={n})")

In [ ]:
# PROMPT ICC SUMMARY
print("\n" + "="*80)
print("PROMPT ICC SUMMARY")
print("="*80)

print(f"""
QUESTION: Does prompt (topic) explain variance in shape metrics?

RESULTS:
""")

for metric in ['log_slope_local', 'auc_linear_k']:
    icc_raw = prompt_icc_results[metric]['icc']
    icc_resid = prompt_icc_resid_results[metric]['icc'] if metric in prompt_icc_resid_results else np.nan
    spread = prompt_spread_results[metric]['spread']
    grand_mean = prompt_spread_results[metric]['grand_mean']
    
    print(f"  {metric}:")
    print(f"    ICC(prompt) raw: {icc_raw:.3f}")
    print(f"    ICC(prompt) after length+fluency control: {icc_resid:.3f}")
    print(f"    Prompt spread: {spread:.4f} (grand mean: {grand_mean:.4f})")
    print()

print("""
INTERPRETATION:
   ICC < 0.05: Prompt is negligible - can safely ignore
   ICC 0.05-0.10: Weak prompt effect
   ICC > 0.10: Prompt matters, consider controlling for it

   If ICC(resid) << ICC(raw): Prompt effect was really just length/vocab
   If ICC(resid) ≈ ICC(raw): Prompt has real effect on memory structure
""")

## 8. Summary and Interpretation

In [ ]:
print("\n" + "="*80)
print("SUMMARY: Within-Essay Stability of Shape Metrics")
print("="*80)

print(f"""
QUESTION: Are shape metrics stable within individual essays (across segments)?

KEY RESULTS:

1. INTRACLASS CORRELATION (ICC)
   Fraction of variance that is between-essay vs within-essay noise:
""")

for metric in ['log_slope_local', 'auc_linear_k', 'gain_128']:
    if metric in icc_results:
        icc = icc_results[metric]['icc']
        interp = "weak" if icc < 0.10 else "moderate" if icc < 0.30 else "STRONG"
        print(f"   {metric}: ICC = {icc:.3f} ({interp})")

print(f"""
2. SPATIAL STABILITY (first-half vs second-half)
   Correlation of metrics from early vs late portions of essay:
""")

for metric in ['log_slope_local', 'auc_linear_k']:
    if metric in spatial_results:
        r = spatial_results[metric]['r_pearson']
        print(f"   {metric}: r = {r:.3f}")

print(f"""
3. WITHIN vs BETWEEN DISTANCES
   Ratio of within-essay to between-essay differences:
""")

for metric in ['log_slope_local', 'auc_linear_k']:
    if metric in distance_results:
        ratio = distance_results[metric]['ratio']
        print(f"   {metric}: ratio = {ratio:.3f} (lower = better signature)")

print(f"""
4. RESIDUALIZED ICC (controlling for length/fluency)
   Does the signature survive after removing confounds?
""")

for metric in ['log_slope_local', 'auc_linear_k']:
    if metric in icc_resid_results:
        icc_raw = icc_results[metric]['icc']
        icc_resid = icc_resid_results[metric]['icc']
        print(f"   {metric}: ICC raw={icc_raw:.3f}, ICC resid={icc_resid:.3f}")

print(f"""
INTERPRETATION:
   ICC < 0.10: Weak signature - mostly noise
   ICC 0.10-0.30: Moderate signature - some stability
   ICC > 0.30: Strong signature - essays have distinctive profiles

   If ICC(resid) ≈ ICC(raw): Signature is real, not confounded
   If ICC(resid) << ICC(raw): Signature was mostly length/fluency
""")

## 9. Save Results

In [ ]:
# Save window-level metrics
df_window_metrics.to_csv(output_dir / 'window_shape_metrics.csv', index=False)

# Save essay-level summary (with prompt_id)
essay_summary = df_window_metrics.groupby('essay_id').agg({
    'log_slope_local': ['mean', 'std'],
    'auc_linear_k': ['mean', 'std'],
    'gain_128': ['mean', 'std'],
    'early_ratio': ['mean', 'std'],
    'token_count': 'first',
    'baseline_nll': 'first',
    'score_bin': 'first',
    'prompt_id': 'first',
}).reset_index()
essay_summary.columns = ['_'.join(col).strip('_') for col in essay_summary.columns]
essay_summary.to_csv(output_dir / 'essay_signature_summary.csv', index=False)

# Save stability report (including prompt ICC)
with open(output_dir / 'stability_report.txt', 'w') as f:
    f.write("WITHIN-ESSAY STABILITY ANALYSIS\n")
    f.write("="*80 + "\n\n")
    
    f.write("QUESTION: Are shape metrics stable within essays (across segments)?\n")
    f.write("This tests whether essays have distinctive 'profiles' that could be signatures.\n\n")
    
    f.write("="*60 + "\n")
    f.write("1. INTRACLASS CORRELATION (ICC) - ESSAY LEVEL\n")
    f.write("="*60 + "\n")
    f.write("ICC = Var(between-essay) / Var(total)\n\n")
    
    for metric, result in icc_results.items():
        f.write(f"{metric}:\n")
        f.write(f"  ICC = {result['icc']:.4f}\n")
        f.write(f"  Var(between) = {result['var_between']:.6f} ({result['pct_between']:.1f}%)\n")
        f.write(f"  Var(within) = {result['var_within']:.6f} ({100-result['pct_between']:.1f}%)\n\n")
    
    f.write("="*60 + "\n")
    f.write("2. SPATIAL STABILITY (first-half vs second-half)\n")
    f.write("="*60 + "\n\n")
    
    for metric, result in spatial_results.items():
        f.write(f"{metric}: r = {result['r_pearson']:.4f} (p = {result['p_pearson']:.4f})\n")
    
    f.write("\n" + "="*60 + "\n")
    f.write("3. WITHIN vs BETWEEN DISTANCES\n")
    f.write("="*60 + "\n\n")
    
    for metric, result in distance_results.items():
        f.write(f"{metric}:\n")
        f.write(f"  Within-essay mean: {result['within_mean']:.4f}\n")
        f.write(f"  Between-essay mean: {result['between_mean']:.4f}\n")
        f.write(f"  Ratio: {result['ratio']:.4f}\n")
        f.write(f"  p-value: {result['pval']:.4f}\n\n")
    
    f.write("="*60 + "\n")
    f.write("4. BOOTSTRAP RANK STABILITY\n")
    f.write("="*60 + "\n\n")
    
    for metric, result in rank_results.items():
        f.write(f"{metric}:\n")
        f.write(f"  Mean r = {result['pearson_mean']:.4f}\n")
        f.write(f"  95% CI = [{result['pearson_ci'][0]:.4f}, {result['pearson_ci'][1]:.4f}]\n\n")
    
    f.write("="*60 + "\n")
    f.write("5. RESIDUALIZED ICC (controlling for length/fluency)\n")
    f.write("="*60 + "\n\n")
    
    for metric in icc_resid_results:
        f.write(f"{metric}:\n")
        f.write(f"  ICC (raw) = {icc_results[metric]['icc']:.4f}\n")
        f.write(f"  ICC (residualized) = {icc_resid_results[metric]['icc']:.4f}\n\n")
    
    f.write("="*60 + "\n")
    f.write("6. PROMPT ICC ANALYSIS\n")
    f.write("="*60 + "\n")
    f.write("Does topic explain variance in shape metrics?\n\n")
    
    f.write("Unconditional Prompt ICC:\n")
    for metric in prompt_icc_results:
        f.write(f"  {metric}: ICC = {prompt_icc_results[metric]['icc']:.4f}\n")
    
    f.write("\nPrompt ICC after length+fluency control:\n")
    for metric in prompt_icc_resid_results:
        icc_raw = prompt_icc_results[metric]['icc']
        icc_resid = prompt_icc_resid_results[metric]['icc']
        f.write(f"  {metric}: ICC raw={icc_raw:.4f}, ICC resid={icc_resid:.4f}\n")
    
    f.write("\nPrompt fixed-effect spread (min-max):\n")
    for metric in prompt_spread_results:
        spread = prompt_spread_results[metric]['spread']
        grand = prompt_spread_results[metric]['grand_mean']
        f.write(f"  {metric}: spread={spread:.4f} (grand mean={grand:.4f})\n")

print(f"\nSaved to {output_dir}/")
print(f"  - window_shape_metrics.csv")
print(f"  - essay_signature_summary.csv")
print(f"  - stability_report.txt")
print(f"  - icc_variance_components.png")
print(f"  - spatial_stability.png")
print(f"  - within_vs_between_distances.png")
print(f"  - rank_stability.png")